<a href="https://colab.research.google.com/github/evgeniy-borisov/vairl/blob/main/notebooks/hybrid-agent-dag-fsm-bt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Гибридный агент: DAG, FSM и Behavior Tree

Минимальные исполнители трёх представлений логики агента, визуализация графов и **конвертация в ограниченных случаях**.

**VAIRL** — Virtual AI Research Lab

Статья: [Гибридный агент: DAG, конечный автомат и деревья поведения](https://evgeniy-borisov.github.io/vairl/blog/2026/06/26/hybrid-agent-dag-fsm-behavior-tree/)

## 1. Установка зависимостей

In [ ]:
!pip install -q networkx matplotlib

import networkx as nx
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Any, Callable, Iterable

print("Готово: networkx", nx.__version__)

## 2. Общая визуализация графов

Единый слой отрисовки для DAG, FSM и BT — layered layout для workflow, spring layout для автоматов, top-down для деревьев.

In [ ]:
def draw_graph(
    G: nx.DiGraph,
    *,
    layout: str = "spring",
    title: str = "",
    highlight: str | None = None,
    node_labels: dict[str, str] | None = None,
    figsize=(10, 6),
):
    """layout: spring | layered | tree"""
    plt.figure(figsize=figsize)
    labels = node_labels or {n: n for n in G.nodes}

    if layout == "layered":
        try:
            pos = nx.nx_agraph.graphviz_layout(G, prog="dot")
        except Exception:
            layers = {}
            for node in nx.topological_sort(G):
                preds = list(G.predecessors(node))
                rank = max((layers[p] for p in preds), default=-1) + 1
                layers[node] = rank
            by_layer: dict[int, list[str]] = {}
            for n, r in layers.items():
                by_layer.setdefault(r, []).append(n)
            pos = {}
            for r, nodes in sorted(by_layer.items()):
                for i, n in enumerate(nodes):
                    pos[n] = (i - len(nodes) / 2, -r)
    elif layout == "tree":
        pos = nx.nx_agraph.graphviz_layout(G, prog="dot") if _has_graphviz() else nx.spring_layout(G, seed=0)
    else:
        pos = nx.spring_layout(G, seed=0, k=1.5)

    colors = ["#ff6b6b" if n == highlight else "#4dabf7" for n in G.nodes]
    nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=1200, alpha=0.9)
    nx.draw_networkx_labels(G, pos, labels={n: labels[n] for n in G.nodes}, font_size=8)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=16, edge_color="#495057")
    edge_labels = nx.get_edge_attributes(G, "label")
    if edge_labels:
        nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=7)
    plt.title(title)
    plt.axis("off")
    plt.tight_layout()
    plt.show()


def _has_graphviz() -> bool:
    try:
        nx.nx_agraph.graphviz_layout(nx.DiGraph([("a", "b")]), prog="dot")
        return True
    except Exception:
        return False

## 3. Конечный автомат (FSM)

Состояния, события, guard-функции. Циклы допустимы — типично для долгоживущей сессии агента.

In [ ]:
@dataclass
class Transition:
    event: str
    target: str
    guard: Callable[[dict[str, Any]], bool] = lambda ctx: True


class FiniteStateMachine:
    def __init__(self, initial: str):
        self.initial = initial
        self.state = initial
        self._transitions: dict[str, list[Transition]] = {}
        self.trace: list[str] = [initial]

    def on(self, source: str, event: str, target: str, guard=None):
        g = guard or (lambda ctx: True)
        self._transitions.setdefault(source, []).append(Transition(event, target, g))
        return self

    def dispatch(self, event: str, ctx: dict[str, Any] | None = None) -> bool:
        ctx = ctx or {}
        for tr in self._transitions.get(self.state, []):
            if tr.event == event and tr.guard(ctx):
                self.state = tr.target
                self.trace.append(self.state)
                return True
        return False

    def to_graph(self) -> nx.DiGraph:
        G = nx.DiGraph()
        for src, transitions in self._transitions.items():
            G.add_node(src)
            for tr in transitions:
                G.add_edge(src, tr.target, label=tr.event)
        return G


# Пример: цикл plan → execute → reflect
fsm = (
    FiniteStateMachine("idle")
    .on("idle", "goal", "planning")
    .on("planning", "plan_ready", "executing")
    .on("executing", "step_done", "reflecting")
    .on("reflecting", "continue", "executing")
    .on("reflecting", "done", "idle")
    .on("executing", "error", "recovering")
    .on("recovering", "replan", "planning")
)

events = ["goal", "plan_ready", "step_done", "continue", "step_done", "done"]
for ev in events:
    fsm.dispatch(ev)

print("Trace:", " → ".join(fsm.trace))
draw_graph(fsm.to_graph(), layout="spring", title="FSM агента", highlight=fsm.state)

## 4. Behavior Tree (BT)

Классические узлы: **Sequence**, **Selector**, **Parallel**, **Condition**, **Action**. Дерево переоценивается каждый **tick**.

In [ ]:
class Status(Enum):
    SUCCESS = auto()
    FAILURE = auto()
    RUNNING = auto()


class BTNode:
    name: str = "node"

    def tick(self, ctx: dict[str, Any]) -> Status:
        raise NotImplementedError


class Sequence(BTNode):
    def __init__(self, name: str, children: list[BTNode]):
        self.name = name
        self.children = children

    def tick(self, ctx: dict[str, Any]) -> Status:
        for child in self.children:
            st = child.tick(ctx)
            if st != Status.SUCCESS:
                return st
        return Status.SUCCESS


class Selector(BTNode):
    def __init__(self, name: str, children: list[BTNode]):
        self.name = name
        self.children = children
    
    def tick(self, ctx: dict[str, Any]) -> Status:
        for child in self.children:
            st = child.tick(ctx)
            if st != Status.FAILURE:
                return st
        return Status.FAILURE


class Condition(BTNode):
    def __init__(self, name: str, pred: Callable[[dict], bool]):
        self.name = name
        self.pred = pred

    def tick(self, ctx: dict[str, Any]) -> Status:
        return Status.SUCCESS if self.pred(ctx) else Status.FAILURE


class Action(BTNode):
    def __init__(self, name: str, fn: Callable[[dict], None] | None = None):
        self.name = name
        self.fn = fn or (lambda ctx: ctx.setdefault("log", []).append(name))

    def tick(self, ctx: dict[str, Any]) -> Status:
        self.fn(ctx)
        return Status.SUCCESS


def bt_to_graph(root: BTNode, parent: str | None = None, G: nx.DiGraph | None = None, nid: int = 0):
    G = G or nx.DiGraph()
    node_id = f"{root.__class__.__name__}:{root.name}#{nid}"
    G.add_node(node_id, label=f"{root.__class__.__name__}\n{root.name}")
    if parent:
        G.add_edge(parent, node_id)
    children = getattr(root, "children", [])
    for i, ch in enumerate(children):
        bt_to_graph(ch, node_id, G, nid * 10 + i + 1)
    return G


ctx = {"has_goal": True, "log": []}
bt = Selector(
    "main",
    [
        Sequence(
            "handle_task",
            [
                Condition("has_goal", lambda c: c.get("has_goal", False)),
                Selector(
                    "strategy",
                    [Action("plan"), Action("react")],
                ),
                Action("execute"),
            ],
        ),
        Action("wait"),
    ],
)

status = bt.tick(ctx)
print("BT status:", status.name, "| log:", ctx["log"])
G_bt = bt_to_graph(bt)
draw_graph(G_bt, layout="tree", title="Behavior Tree", node_labels={n: G_bt.nodes[n]["label"] for n in G_bt.nodes})

## 5. DAG — workflow без циклов

Узлы запускаются после готовности всех предшественников (topological order). Параллельные ветки — естественны.

In [ ]:
class DAGWorkflow:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.handlers: dict[str, Callable[[dict], None]] = {}

    def add_node(self, name: str, fn: Callable[[dict], None] | None = None):
        self.graph.add_node(name)
        self.handlers[name] = fn or (lambda ctx: ctx.setdefault("dag_log", []).append(name))
        return self

    def add_edge(self, src: str, dst: str):
        self.graph.add_edge(src, dst)
        return self

    def run(self, ctx: dict[str, Any] | None = None) -> list[str]:
        ctx = ctx or {}
        if not nx.is_directed_acyclic_graph(self.graph):
            raise ValueError("Graph must be acyclic (DAG)")
        order = list(nx.topological_sort(self.graph))
        for name in order:
            self.handlers[name](ctx)
        return order


dag = (
    DAGWorkflow()
    .add_node("parse_intent")
    .add_node("retrieve")
    .add_node("validate_schema")
    .add_node("synthesize")
    .add_node("execute_tools")
    .add_node("critic")
    .add_edge("parse_intent", "retrieve")
    .add_edge("parse_intent", "validate_schema")
    .add_edge("retrieve", "synthesize")
    .add_edge("validate_schema", "synthesize")
    .add_edge("synthesize", "execute_tools")
    .add_edge("execute_tools", "critic")
)

ctx_dag: dict[str, Any] = {}
order = dag.run(ctx_dag)
print("Topological order:", order)
print("Executed:", ctx_dag.get("dag_log"))
draw_graph(dag.graph, layout="layered", title="Agent pipeline (DAG)", highlight="critic")

## 6. Гибридный оркестратор

Meta-controller переключает активный исполнитель: `dag` | `fsm` | `bt`. Один реестр листовых действий — три представления.

In [ ]:
class HybridOrchestrator:
    def __init__(self):
        self.mode = "dag"
        self.executors: dict[str, Any] = {}
        self.history: list[str] = []

    def register(self, mode: str, executor: Any):
        self.executors[mode] = executor
        return self

    def switch(self, mode: str):
        if mode not in self.executors:
            raise KeyError(f"Unknown mode: {mode}")
        self.mode = mode
        self.history.append(f"switch→{mode}")

    def step(self, ctx: dict[str, Any], **kwargs) -> Any:
        ex = self.executors[self.mode]
        if self.mode == "fsm":
            ok = ex.dispatch(kwargs.get("event", ""), ctx)
            self.history.append(f"fsm:{ex.state}")
            return ok
        if self.mode == "bt":
            st = ex.tick(ctx)
            self.history.append(f"bt:{st.name}")
            return st
        if self.mode == "dag":
            order = ex.run(ctx)
            self.history.append(f"dag:{len(order)} nodes")
            return order
        raise ValueError(self.mode)


hybrid = (
    HybridOrchestrator()
    .register("dag", dag)
    .register("fsm", fsm)
    .register("bt", bt)
)

ctx_h: dict[str, Any] = {"has_goal": True, "log": []}
hybrid.switch("dag")
hybrid.step(ctx_h)
hybrid.switch("fsm")
hybrid.step(ctx_h, event="goal")
hybrid.switch("bt")
hybrid.step(ctx_h)

print("Hybrid history:")
for line in hybrid.history:
    print(" ", line)

## 7. Конвертация в ограниченных случаях

Полная эквивалентность между моделями **не гарантируется**. Ниже — осмысленные преобразования топологии.

In [ ]:
def linear_dag_to_bt_sequence(dag_wf: DAGWorkflow) -> Sequence:
    """DAG без параллелизма (цепочка) → BT Sequence."""
    order = list(nx.topological_sort(dag_wf.graph))
    indeg = dict(dag_wf.graph.in_degree())
    outdeg = dict(dag_wf.graph.out_degree())
    if not all(indeg[n] <= 1 for n in order) or not all(outdeg[n] <= 1 for n in order):
        raise ValueError("Only linear chains supported (in/out degree ≤ 1)")
    return Sequence("from_dag", [Action(n) for n in order])


def acyclic_fsm_to_dag(fsm_obj: FiniteStateMachine, max_depth: int = 20) -> nx.DiGraph:
    """Развёртка путей FSM без циклов в DAG (BFS по парам state×depth)."""
    G = nx.DiGraph()
    start = fsm_obj.initial
    G.add_node(f"{start}@0")
    queue = [(start, 0)]
    seen = set()
    while queue:
        state, depth = queue.pop(0)
        if depth >= max_depth:
            continue
        key = (state, depth)
        if key in seen:
            continue
        seen.add(key)
        for tr in fsm_obj._transitions.get(state, []):
            src = f"{state}@{depth}"
            dst = f"{tr.target}@{depth + 1}"
            G.add_edge(src, dst, label=tr.event)
            if tr.target != state:  # не углубляем бесконечный self-loop
                queue.append((tr.target, depth + 1))
    return G


def flat_fsm_to_bt_selector(fsm_obj: FiniteStateMachine) -> Selector:
    """Каждое состояние — ветка Selector с Condition state==X и Action enter_X."""
    states = set(fsm_obj._transitions.keys())
    for transitions in fsm_obj._transitions.values():
        for tr in transitions:
            states.add(tr.target)
    branches = []
    for st in sorted(states):
        branches.append(
            Sequence(
                f"state_{st}",
                [
                    Condition(f"is_{st}", lambda c, s=st: c.get("fsm_state") == s),
                    Action(f"run_{st}", lambda c, s=st: c.setdefault("bt_fsm_log", []).append(s)),
                ],
            )
        )
    return Selector("fsm_as_bt", branches)


def bt_linear_to_dag(root: BTNode) -> DAGWorkflow:
    """BT без Selector/Parallel — только цепочка Sequence → DAG."""
    if not isinstance(root, Sequence):
        raise ValueError("Root must be Sequence")
    wf = DAGWorkflow()
    prev = None
    for ch in root.children:
        if isinstance(ch, Action):
            wf.add_node(ch.name)
            if prev:
                wf.add_edge(prev, ch.name)
            prev = ch.name
        else:
            raise ValueError(f"Non-linear child: {type(ch).__name__}")
    return wf


# --- демо конвертаций ---

linear = DAGWorkflow()
for step in ["retrieve", "plan", "execute", "critic"]:
    linear.add_node(step)
for a, b in zip(["retrieve", "plan", "execute"], ["plan", "execute", "critic"]):
    linear.add_edge(a, b)

bt_from_dag = linear_dag_to_bt_sequence(linear)
ctx_conv: dict[str, Any] = {"log": []}
bt_from_dag.tick(ctx_conv)
print("Linear DAG → BT log:", ctx_conv["log"])

G_unrolled = acyclic_fsm_to_dag(fsm, max_depth=5)
print(f"FSM unroll → DAG: {G_unrolled.number_of_nodes()} nodes, {G_unrolled.number_of_edges()} edges")
draw_graph(G_unrolled, layout="layered", title="FSM развёрнут в DAG (max_depth=5)")

bt_fsm = flat_fsm_to_bt_selector(fsm)
ctx_fsm = {"fsm_state": "executing", "bt_fsm_log": []}
bt_fsm.tick(ctx_fsm)
print("Flat FSM → BT (state=executing):", ctx_fsm["bt_fsm_log"])

roundtrip = bt_linear_to_dag(bt_from_dag)
print("BT → DAG nodes:", list(roundtrip.graph.nodes))

## 8. Сводка: когда конвертировать

| Направление | Условие | Риск |
|-------------|---------|------|
| Линейный DAG → BT | in/out degree ≤ 1 | низкий |
| Fork-join DAG → BT | Parallel + join Sequence | средний |
| Ацикличный FSM → DAG | ограниченная глубина | циклы теряются |
| Плоский FSM → BT | мало состояний | tick-семантика другая |
| BT → DAG | только Sequence of Action | Selector ломает ацикличность |

**Практика:** храните каноническое описание узлов (`Action`/`Condition`) и генерируйте представления; для продакшена — вложенность (FSM снаружи, DAG/BT внутри), а не слепая конвертация.